# 23 — Throughput Objective

**Engineering statement:** Verification allocation specifies decoding throughput.

Notebook 23 unifies the previous notebooks into one systems objective. Notebook 07 framed verification as a scarce resource. Notebook 13 showed how confidence schedules that resource. Notebook 17 showed how draft architecture shapes the confidence profile available to the scheduler. This notebook asks:

> **Which verification policy maximizes useful accepted tokens per unit of serving capacity?**

## Repository roadmap

```text
00 Context
        ↓
07 Verification Resource
        ↓
13 Confidence Scheduling
        ↓
17 Semi-Autoregressive Drafting
        ↓
23 Throughput Objective
        ↓
29 Hardware-Aware Scheduling
        ↓
37 Operating Regimes
        ↓
43 Resource Allocation
        ↓
49 Adaptive AI Infrastructure
```

## Start here

```text
draft quality
        ↓
conditional confidence
        ↓
prefix survival
        ↓
verification allocation
        ↓
expected accepted tokens
        ↓
steps per second
        ↓
throughput objective
```

Throughput is not just draft speed, verifier speed, or accepted prefix length. It is the interaction between all three.

In [1]:
from pathlib import Path
import json
import math
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, Rectangle

FIG_DIR = Path("../figures") if Path.cwd().name == "notebooks" else Path("figures")
RESULT_DIR = Path("../results/23_throughput_objective") if Path.cwd().name == "notebooks" else Path("results/23_throughput_objective")
FIG_DIR.mkdir(parents=True, exist_ok=True)
RESULT_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    "figure.dpi": 160,
    "savefig.dpi": 160,
    "font.size": 12,
    "axes.titlesize": 18,
    "axes.labelsize": 13,
    "legend.fontsize": 11,
})

rng = np.random.default_rng(23)

FIGURES = []

def savefig(name: str):
    path = FIG_DIR / name
    plt.tight_layout()
    plt.savefig(path, bbox_inches="tight")
    plt.close()
    FIGURES.append(str(path))
    return path

def draw_box(ax, x, y, text, w=1.8, h=0.62, fs=13):
    patch = FancyBboxPatch(
        (x-w/2, y-h/2), w, h,
        boxstyle="round,pad=0.05,rounding_size=0.08",
        linewidth=1.5, edgecolor="black", facecolor="white"
    )
    ax.add_patch(patch)
    ax.text(x, y, text, ha="center", va="center", fontsize=fs)
    return patch

def arrow(ax, x0, y0, x1, y1):
    ax.annotate("", xy=(x1, y1), xytext=(x0, y0),
                arrowprops=dict(arrowstyle="->", lw=2, color="black"))

def prefix_survival(conf):
    return np.cumprod(np.asarray(conf, dtype=float))

requests = {
    "chat": np.array([0.88, 0.78, 0.66, 0.54, 0.42, 0.34, 0.28, 0.23]),
    "instruction": np.array([0.92, 0.86, 0.79, 0.70, 0.60, 0.48, 0.38, 0.31]),
    "code": np.array([0.96, 0.91, 0.85, 0.78, 0.68, 0.58, 0.48, 0.39]),
    "math": np.array([0.95, 0.88, 0.80, 0.70, 0.61, 0.50, 0.40, 0.32]),
}

architectures = {
    "parallel": np.array([0.94, 0.77, 0.50, 0.23, 0.07, 0.01, 0.00, 0.00]),
    "autoregressive": np.array([0.86, 0.74, 0.63, 0.53, 0.45, 0.38, 0.31, 0.26]),
    "semi_autoregressive": np.array([0.94, 0.86, 0.75, 0.63, 0.50, 0.36, 0.24, 0.13]),
}

ell_values = np.arange(1, 9)

## 1. The objective pipeline

A useful serving objective has to preserve the causal chain from drafting to verification. The draft model determines a confidence profile. Conditional confidence compounds into prefix survival. The scheduler selects a verification length. Hardware throughput then converts accepted tokens into system throughput.

In [2]:
fig, ax = plt.subplots(figsize=(12, 3.8))
ax.set_xlim(0, 12)
ax.set_ylim(0, 3)
ax.axis("off")
ax.set_title("Throughput objective: from draft quality to serving capacity", pad=18)

labels = [
    "Draft\nquality", "Conditional\nconfidence", "Prefix\nsurvival",
    "Verification\nallocation", "Accepted\ntokens", "Steps/sec", "Throughput"
]
xs = np.linspace(0.9, 11.1, len(labels))
y = 1.55
for i, (x, label) in enumerate(zip(xs, labels)):
    draw_box(ax, x, y, label, w=1.35, h=0.75, fs=12)
    if i < len(xs)-1:
        arrow(ax, x+0.72, y, xs[i+1]-0.72, y)

ax.text(6, 0.35, "Throughput is the product of accepted useful work and serving speed.",
        ha="center", va="center", fontsize=14)
savefig("23_objective_overview.png")

PosixPath('../figures/23_objective_overview.png')

## 2. Expected accepted length

Speculative decoding accepts a continuous prefix. For a scheduled prefix length \(\ell\), the expected accepted length is

\[
\tau(\ell)=1+\sum_{j=1}^{\ell}a_j,
\]

where \(a_j\) is the probability that the draft prefix survives through position \(j\). The leading \(1\) represents the target model's fallback token when the speculative prefix stops.

In [3]:
def tau_from_survival(a, ell):
    ell = int(ell)
    return 1.0 + float(np.sum(a[:ell]))

def sps_from_batch(B, base=230.0, curvature=0.055, floor=35.0):
    # simple decreasing throughput curve for target verification as batch expands
    return floor + base / (1.0 + curvature * B**1.35)

def throughput_for(conf, ell, active_requests=24, draft_cost=0.35, verify_cost=1.0):
    a = prefix_survival(conf)
    tau = tau_from_survival(a, ell)
    B = active_requests * (1 + ell)
    sps = sps_from_batch(B)
    draft_penalty = 1.0 / (1.0 + draft_cost / 6.0)
    return tau * sps * draft_penalty, tau, B, sps

rows = []
for name, conf in requests.items():
    a = prefix_survival(conf)
    for ell in ell_values:
        theta, tau, B, sps = throughput_for(conf, ell)
        rows.append({"request": name, "ell": ell, "tau": tau, "B": B, "SPS": sps, "theta": theta})
obj = pd.DataFrame(rows)
obj.to_csv(RESULT_DIR / "throughput_objective_table.csv", index=False)
obj.head()

,request,ell,tau,B,SPS,theta
0,chat,1,1.880000,48,55.473865,98.542551
1,chat,2,2.566400,72,47.305142,114.712361
2,chat,3,3.019424,96,43.491069,124.079980
3,chat,4,3.264057,120,41.343432,127.509276
4,chat,5,3.366803,144,39.989429,127.215612


## 3. Tradeoff surface

Adding verification tokens usually increases expected accepted length at first. Eventually the marginal accepted token becomes too unlikely, and the larger verification batch reduces steps per second. The optimum appears where those two effects balance.

In [4]:
fig, ax = plt.subplots(figsize=(9, 5.2))
for name in requests:
    sub = obj[obj["request"] == name]
    ax.plot(sub["ell"], sub["theta"], marker="o", label=name)
    best = sub.loc[sub["theta"].idxmax()]
    ax.scatter([best["ell"]], [best["theta"]], s=110, zorder=5)
    ax.annotate(f"best ℓ={int(best['ell'])}", (best["ell"], best["theta"]),
                xytext=(5, 5), textcoords="offset points", fontsize=10)
ax.set_title("Throughput objective balances accepted length and serving speed")
ax.set_xlabel("Scheduled verification length ℓ")
ax.set_ylabel("Expected throughput Θ")
ax.legend(title="request")
ax.grid(True, alpha=0.3)
savefig("23_tradeoff_surface.png")

PosixPath('../figures/23_tradeoff_surface.png')

## 4. Budget allocation across requests

The scheduler should not allocate the same prefix length to every request. High-survival requests justify longer verification. Low-survival requests should stop earlier, preserving verification capacity for other prompts in the batch.

In [5]:
threshold = 0.35
alloc = []
for name, conf in requests.items():
    a = prefix_survival(conf)
    scheduled = int(np.sum(a >= threshold))
    alloc.append((name, scheduled, a))

fig, ax = plt.subplots(figsize=(11, 4.6))
ax.set_title("Verification budget allocation follows prefix survival")
ax.set_xlim(0, 8)
ax.set_ylim(-0.7, len(alloc)-0.3)
ax.set_xlabel("Draft position")
ax.set_yticks(range(len(alloc)))
ax.set_yticklabels([x[0] for x in alloc])

for row, (name, scheduled, a) in enumerate(alloc):
    for j in range(8):
        alpha = 1.0 if j < scheduled else 0.18
        rect = Rectangle((j, row-0.3), 1, 0.6, facecolor=f"C{row}", edgecolor="black", alpha=alpha, lw=1.0)
        ax.add_patch(rect)
        if j < scheduled:
            ax.text(j+0.5, row, f"{a[j]:.2f}", ha="center", va="center", fontsize=10)
    ax.text(8.15, row, f"ℓ={scheduled}", va="center", fontsize=12)
ax.axvline(0, color="black", lw=1)
ax.grid(False)
savefig("23_budget_allocation.png")

pd.DataFrame([{"request": n, "scheduled_length": s} for n, s, _ in alloc]).to_csv(
    RESULT_DIR / "budget_allocation.csv", index=False
)

## 5. Operating regions

The same scheduler can behave differently depending on confidence quality and system load. A low-load system can afford longer verification. A high-load system should reserve verification capacity for prefixes with strong survival probability.

In [6]:
confidence_floor = np.linspace(0.2, 0.8, 61)
load = np.linspace(0.2, 1.0, 61)
Z = np.zeros((len(load), len(confidence_floor)))

for i, L in enumerate(load):
    for j, floor in enumerate(confidence_floor):
        # heuristic operating region score: longer schedules require confidence; high load pushes shorter schedules
        Z[i, j] = 8 * floor * (1.15 - L)

fig, ax = plt.subplots(figsize=(8.8, 5.2))
cs = ax.contourf(confidence_floor, load, Z, levels=[-10, 1.5, 3.5, 5.5, 10], alpha=0.75)
ax.contour(confidence_floor, load, Z, levels=[1.5, 3.5, 5.5], colors="black", linewidths=1, alpha=0.5)
ax.text(0.30, 0.85, "short\nprefix", ha="center", va="center", fontsize=13)
ax.text(0.50, 0.58, "adaptive\nprefix", ha="center", va="center", fontsize=13)
ax.text(0.70, 0.34, "long\nprefix", ha="center", va="center", fontsize=13)
ax.set_title("Operating regions depend on confidence and system load")
ax.set_xlabel("Prefix survival quality")
ax.set_ylabel("Serving load")
fig.colorbar(cs, ax=ax, label="illustrative scheduled length")
savefig("23_operating_regions.png")

PosixPath('../figures/23_operating_regions.png')

## 6. Optimal schedule selection

The scheduler can be viewed as a discrete optimizer. It evaluates candidate prefix lengths, converts each into expected accepted tokens, accounts for hardware speed, and selects the length with maximum objective value.

In [7]:
name = "code"
sub = obj[obj["request"] == name].copy()
best = sub.loc[sub["theta"].idxmax()]

fig, ax = plt.subplots(figsize=(9, 5.0))
ax.plot(sub["ell"], sub["theta"], marker="o", lw=2)
ax.axvline(best["ell"], ls="--", lw=1.5)
ax.scatter([best["ell"]], [best["theta"]], s=180, zorder=5)
ax.annotate(f"optimal schedule\nℓ*={int(best['ell'])}",
            (best["ell"], best["theta"]), xytext=(15, -40),
            textcoords="offset points", arrowprops=dict(arrowstyle="->", lw=1.5), fontsize=12)
ax.set_title("Optimal scheduled prefix maximizes useful throughput")
ax.set_xlabel("Scheduled verification length ℓ")
ax.set_ylabel("Expected throughput Θ")
ax.grid(True, alpha=0.3)
savefig("23_optimal_schedule.png")

PosixPath('../figures/23_optimal_schedule.png')

## 7. Scaling with active batch size

Hardware-aware scheduling becomes important because verification length changes the target-model batch. As the number of active requests increases, the same prefix length can become too expensive.

In [8]:
active_grid = np.arange(4, 65, 4)
ells = np.arange(1, 9)
conf = requests["math"]
scale_rows = []
for R in active_grid:
    for ell in ells:
        theta, tau, B, sps = throughput_for(conf, ell, active_requests=R)
        scale_rows.append({"active_requests": R, "ell": ell, "theta": theta, "B": B, "SPS": sps})
scale = pd.DataFrame(scale_rows)
scale.to_csv(RESULT_DIR / "batch_scaling.csv", index=False)

fig, ax = plt.subplots(figsize=(9, 5.2))
for ell in [1, 2, 4, 6, 8]:
    sub = scale[scale["ell"] == ell]
    ax.plot(sub["active_requests"], sub["theta"], marker="o", label=f"ℓ={ell}")
ax.set_title("Throughput scaling depends on verification length and active batch")
ax.set_xlabel("Active requests R")
ax.set_ylabel("Expected throughput Θ")
ax.legend(title="scheduled length")
ax.grid(True, alpha=0.3)
savefig("23_scaling.png")

PosixPath('../figures/23_scaling.png')

## 8. Architecture summary

Notebook 23 completes the first systems loop. Drafting produces candidate blocks. Confidence converts those candidates into prefix survival. The scheduler allocates target-model verification. The throughput objective selects the operating point.

In [9]:
fig, ax = plt.subplots(figsize=(12, 4.2))
ax.set_xlim(0, 12)
ax.set_ylim(0, 4)
ax.axis("off")
ax.set_title("Confidence-scheduled serving system", pad=18)

nodes = [
    (1.1, 2.4, "Prompt"),
    (3.0, 2.4, "Draft\nmodel"),
    (5.0, 2.4, "Confidence\nprofile"),
    (7.1, 2.4, "Verification\nscheduler"),
    (9.3, 2.4, "Target\nverification"),
    (11.0, 2.4, "Accepted\ntokens"),
]
for i, (x, y, label) in enumerate(nodes):
    draw_box(ax, x, y, label, w=1.35, h=0.72, fs=12)
    if i < len(nodes)-1:
        arrow(ax, x+0.72, y, nodes[i+1][0]-0.72, y)

ax.text(5.9, 0.85, "Objective: maximize accepted useful work under verification and hardware constraints.",
        ha="center", fontsize=14)
savefig("23_architecture_summary.png")

PosixPath('../figures/23_architecture_summary.png')

## Key equations

Expected accepted length:

\[
\tau(\ell)=1+\sum_{j=1}^{\ell}a_j
\]

Target-model verification batch:

\[
B=\sum_{r=1}^{R}(1+\ell_r)
\]

Hardware speed as a function of batch size:

\[
S=S(B)
\]

Overall throughput objective:

\[
\Theta(\ell)=\tau(\ell)S(B)
\]

Optimal scheduled prefix:

\[
\ell^*=\arg\max_{\ell}\Theta(\ell)
\]

## Engineering summary

Draft quality specifies confidence. Confidence specifies schedulable prefixes. Verification allocation specifies throughput.

Notebook 23 turns this chain into an optimization problem: select the schedule that maximizes useful accepted tokens while respecting target-model verification capacity and serving load.

In [10]:
manifest = {
    "notebook": "23_throughput_objective.ipynb",
    "title": "23 — Throughput Objective",
    "engineering_statement": "Verification allocation specifies decoding throughput.",
    "figures": [Path(f).name for f in FIGURES],
    "outputs": [
        "throughput_objective_table.csv",
        "budget_allocation.csv",
        "batch_scaling.csv",
    ],
    "next_notebook": "29_hardware_aware_scheduling.ipynb",
}
(RESULT_DIR / "manifest.json").write_text(json.dumps(manifest, indent=2))
manifest

{'notebook': '23_throughput_objective.ipynb',
 'title': '23 — Throughput Objective',
 'engineering_statement': 'Verification allocation specifies decoding throughput.',
 'figures': ['23_objective_overview.png',
  '23_tradeoff_surface.png',
  '23_budget_allocation.png',
  '23_operating_regions.png',
  '23_optimal_schedule.png',
  '23_scaling.png',
  '23_architecture_summary.png'],
 'outputs': ['throughput_objective_table.csv',
  'budget_allocation.csv',
  'batch_scaling.csv'],
 'next_notebook': '29_hardware_aware_scheduling.ipynb'}

## Download artifacts

The final cell packages Notebook 23 outputs into a reproducible archive containing generated figures, CSV tables, and a manifest.

In [11]:
archive = RESULT_DIR / "23_throughput_objective_artifacts.zip"
with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as zf:
    for fig in FIGURES:
        zf.write(fig, arcname=f"figures/{Path(fig).name}")
    for p in RESULT_DIR.glob("*.csv"):
        zf.write(p, arcname=f"results/{p.name}")
    zf.write(RESULT_DIR / "manifest.json", arcname="manifest.json")
print(f"Wrote {archive}")

Wrote ../results/23_throughput_objective/23_throughput_objective_artifacts.zip


## Next notebook

**29 — Hardware-Aware Scheduling** should ask how GPU batch behavior, verification kernels, and serving load change the optimal schedule. Notebook 23 gives the objective; Notebook 29 studies the hardware constraints that shape that objective.